# Olist E-Commerce - SQL + pandas interoperability demo

**Author:** Diego Ospina

> A short companion notebook showing SQL as a complementary tool inside a Python data workflow. We load the raw CSVs into an **in-memory SQLite** database, run real SQL analysis queries (equivalent to `sql/queries_analysis.sql`), and pull the results back into **pandas** for charting. This demonstrates the ETL + SQL + DataFrame blend that is common in analytics.

> The `.sql` files in `sql/` define the schema and the full query set (PostgreSQL dialect); here we use SQLite syntax so the notebook runs with zero external dependencies.

## 1. Create an in-memory SQLite database and load the raw CSVs

In [1]:
import pandas as pd
import sqlite3
import os

RAW = os.path.join('..', 'data', 'raw')

con = sqlite3.connect(':memory:')   # base de datos en memoria
cur = con.cursor()

# Cargar fixtures como tablas SQLite
orders  = pd.read_csv(os.path.join(RAW, 'olist_orders_dataset.csv'))
items   = pd.read_csv(os.path.join(RAW, 'olist_order_items_dataset.csv'))
products= pd.read_csv(os.path.join(RAW, 'olist_products_dataset.csv'))
customers=pd.read_csv(os.path.join(RAW, 'olist_customers_dataset.csv'))
payments= pd.read_csv(os.path.join(RAW, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(RAW, 'olist_order_reviews_dataset.csv'))
trans   = pd.read_csv(os.path.join(RAW, 'product_category_name_translation.csv'))

orders.to_sql('orders', con, if_exists='replace', index=False)
items.to_sql('order_items', con, if_exists='replace', index=False)
products.to_sql('products', con, if_exists='replace', index=False)
customers.to_sql('customers', con, if_exists='replace', index=False)
payments.to_sql('order_payments', con, if_exists='replace', index=False)
reviews.to_sql('order_reviews', con, if_exists='replace', index=False)
trans.to_sql('category_translation', con, if_exists='replace', index=False)
print('tabular loaded:', len(orders), 'orders,', len(items), 'items')

tabular loaded: 99441 orders, 112650 items


## 2. SQL query: monthly sales of delivered orders

In [2]:
q = """
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS purchase_month,
       ROUND(SUM(oi.price), 2)                          AS sales_brl
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY 1
"""
monthly = pd.read_sql_query(q, con)
print(monthly.head().to_string(index=False))
print('...')
print('months:', len(monthly))

purchase_month  sales_brl
       2016-09     134.97
       2016-10   40325.11
       2016-12      10.90
       2017-01  111798.36
       2017-02  234223.40
...
months: 23


## 3. SQL query: on-time vs late delivery (delivered orders)

In [3]:
q = """
SELECT CASE
          WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date THEN 'On time'
          ELSE 'Late' END AS delivery_status,
       COUNT(*) AS orders
FROM orders o
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY 1
"""
deliv = pd.read_sql_query(q, con)
deliv['pct'] = (deliv['orders'] / deliv['orders'].sum() * 100).round(2)
print(deliv.to_string(index=False))

delivery_status  orders   pct
           Late    7826  8.11
        On time   88644 91.89


## 4. SQL query: top categories by sales (with translation)

In [4]:
q = """
SELECT COALESCE(t.product_category_name_english, 'not_specified') AS category_en,
       ROUND(SUM(oi.price), 2) AS sales_brl
FROM order_items oi
JOIN products p  ON p.product_id = oi.product_id
JOIN orders o    ON o.order_id = oi.order_id
LEFT JOIN category_translation t ON t.product_category_name = p.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY sales_brl DESC
LIMIT 8
"""
top_cat = pd.read_sql_query(q, con)
print(top_cat.to_string(index=False))

          category_en  sales_brl
        health_beauty 1233131.72
        watches_gifts 1166176.98
       bed_bath_table 1023434.76
       sports_leisure  954852.55
computers_accessories  888724.61
      furniture_decor  711927.69
           housewares  615628.69
           cool_stuff  610204.10


## 5. SQL query: average review score by punctuality

In [5]:
q = """
SELECT CASE
          WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date THEN 'On time'
          ELSE 'Late' END AS delivery_status,
       ROUND(AVG(r.review_score), 3) AS avg_review,
       COUNT(*)                     AS n_reviews
FROM orders o
JOIN order_reviews r ON r.order_id = o.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY 1
"""
sc = pd.read_sql_query(q, con)
print(sc.to_string(index=False))
con.close()

delivery_status  avg_review  n_reviews
           Late       2.566       7700
        On time       4.294      88653


## Takeaways

- The exact same analytics can be expressed in SQL (`sql/queries_analysis.sql`) and fed back into pandas via `pd.read_sql_query` for charting.
- This pattern (SQL for data querying + pandas for transformation/visualisation) matches a best-practice analytics workflow and is fully reproducible without a server.
- Results agree with the pandas-only notebooks (e.g. ~92% on-time share, top categories dominated by health/beauty, on-time reviews higher than late).